In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import random
import matplotlib.pyplot as plt
from collections import Counter

if torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

class UnsupervisedClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_classes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.LayerNorm(128),
            nn.ReLU(),
            
            nn.Linear(128, 512),
            nn.LayerNorm(512),
            nn.ReLU(),
            
            nn.Linear(512, num_classes)
        )

    def forward(self, x, tau=1.0):
        logits = self.net(x)
        probs = F.gumbel_softmax(logits, tau=tau, hard=False)
        return probs

def uniform_distribution(K):
    
    return torch.full((K,), 1.0 / K, device=device)

def kl_divergence_loss(probs):
    avg_probs = probs.mean(dim=0)
    target = uniform_distribution(probs.size(1))
    
    kl = avg_probs * (torch.log(avg_probs + 1e-10) - torch.log(target))
    return kl.sum()

def entropy_loss(probs):
    return -torch.sum(probs * torch.log(probs + 1e-10), dim=1).mean()

def generate_ip_list(N):
    return [".".join(str(random.randint(0, 255)) for _ in range(4)) for _ in range(N)]

def ip_to_tensor(ip_list):
    parts = [[int(x) for x in ip.split(".")] for ip in ip_list]
    return torch.tensor(parts).float() / 255.0

class IPDataset(torch.utils.data.Dataset):
    def __init__(self, N):
        self.N = N
        self.ip_list = generate_ip_list(N)
        self.ips = ip_to_tensor(self.ip_list)

    def __len__(self):
        return self.N

    def __getitem__(self, idx):
        ip = self.ips[idx]
        x = torch.cat([ip], dim=0)
        return x

def train_unsupervised_classifier(model, dataloader, epochs=20, lr=1e-3, lambda_entropy=1.0, lambda_kl=1.0):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    for epoch in range(epochs):
        total_loss = 0
        for x in dataloader:
            
            x = x.to(device)
            
            probs = model(x)
            ent_loss = entropy_loss(probs)
            kl_loss = kl_divergence_loss(probs)
            loss = - lambda_entropy * ent_loss + lambda_kl * kl_loss

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        print(f"[Epoch {epoch+1}] Loss: {total_loss / len(dataloader):.4f}, Entropy: {-ent_loss.item():.4f}, KL: {kl_loss.item():.4f}")

def predict(model, x):
    with torch.no_grad():
        probs = model(x)
        return torch.argmax(probs, dim=1)

def evaluate_class_distribution(model, dataset, num_classes):
    dataloader = torch.utils.data.DataLoader(dataset, batch_size=256)
    all_preds = []

    with torch.no_grad():
        for x in dataloader:
            x = x.to(device)
            
            preds = predict(model, x)
            all_preds.append(preds.cpu())

    all_preds = torch.cat(all_preds, dim=0)
    unique, counts = torch.unique(all_preds, return_counts=True)
    
    print(f"Used {len(unique)} / {num_classes} classes")

def test_evaluate(model, dataset):
    dataloader = torch.utils.data.DataLoader(dataset, batch_size=256)
    all_preds = []

    with torch.no_grad():
        for x in dataloader:
            x = x.to(device)
            preds = predict(model, x)
            all_preds.append(preds.cpu())

    all_preds = torch.cat(all_preds, dim=0)
    unique, counts = torch.unique(all_preds, return_counts=True)

    print(f"Used {len(unique)} / 4096")
    
    plt.figure(figsize=(12, 5))
    plt.plot(unique.tolist(), counts.tolist())
    plt.title(f"Bucket Usage Distribution (num_buckets={len(unique)})")
    plt.xlabel("Bucket ID")
    plt.ylabel("Count")
    plt.grid(True)
    plt.tight_layout()
    plt.show()

In [ ]:
# pre-train

for memory in [20,40,60,80,100]:
    T = 750
    topK = 1024
    d = 8
    b = 1.08
    memorysize = memory*128/(128+9)

    counter_bit_level1 = 8
    fp_bit_level1 = 8

    counter_bit_level2 = 12
    fp_bit_level2 = 12

    counter_bit_level3 = 20
    fp_bit_level3 = 16
    
    gamma = 4

    W = (counter_bit_level1+fp_bit_level1)*d

    cell_num = int(W / counter_bit_level1 / 2)
    BUCKET_NUM = int(memorysize*1024*8/W)
    K = BUCKET_NUM
    
    ####################################################################
    input_dim = 4
    
    # train
    dataset = IPDataset(N=800000)
    dataloader = torch.utils.data.DataLoader(dataset, batch_size=40960, shuffle=True)

    model = UnsupervisedClassifier(input_dim=input_dim, hidden_dim=512, num_classes=K).to(device)

    train_unsupervised_classifier(model, dataloader, epochs=40)

    evaluate_class_distribution(model, dataset, K)
    
    # test
    dataset = IPDataset(N=700000)
    test_evaluate(model, dataset)

    torch.save(model.state_dict(), "Dispatcher.pt")
    

In [ ]:
# fine-tune

class SpecialIPDataset(torch.utils.data.Dataset):
    def __init__(self, ip_list, labels):
        self.ip_list = ip_list
        self.ips = ip_to_tensor(self.ip_list)
        
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.ip_list)

    def __getitem__(self, idx):
        ip = self.ips[idx]
        x = torch.cat([ip], dim=0)
        return x, self.labels[idx]

heavy_hitters = [] # obtain from detection results
labels = list(range(len(heavy_hitters)))

special_dataset = SpecialIPDataset(heavy_hitters*10, labels*10)
evaluate_dataset = SpecialIPDataset(heavy_hitters, labels)


In [ ]:
from scipy.optimize import linear_sum_assignment

class UnsupervisedClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_classes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.LayerNorm(128),
            nn.ReLU(),
            
            nn.Linear(128, 512),
            nn.LayerNorm(512),
            nn.ReLU(),
            
            nn.Linear(512, num_classes)
        )

    def forward(self, x, tau=0.5):
        logits = self.net(x)
        return logits

model = UnsupervisedClassifier(input_dim=4, hidden_dim=512, num_classes=4096)

model.load_state_dict(torch.load("Dispatcher.pt"))

for name, param in model.named_parameters():
    if "bias" in name:
        param.requires_grad = True
    elif "net.6.weight" in name:
        param.requires_grad = True
    else:
        param.requires_grad = False
    
for name, param in model.named_parameters():
    print(name, param.shape, param.requires_grad)
    
print()

##########################################################################

def super_exponential_conflict_loss(logits, beta=5.0, power=2.0):
    normed = F.normalize(logits, dim=1)
    sim = torch.matmul(normed, normed.T)
    B = sim.size(0)
    mask = ~torch.eye(B, dtype=torch.bool, device=logits.device)
    sim = sim[mask]
    return torch.exp(beta * (sim ** power)).mean()

def kl_divergence_loss(probs):
    avg_probs = probs.mean(dim=0)  # [K]
    target = uniform_distribution(probs.size(1), probs.device)
    kl = avg_probs * (torch.log(avg_probs + 1e-10) - torch.log(target))
    return kl.sum()

def entropy_loss(probs):
    
    return -torch.sum(probs * torch.log(probs + 1e-10), dim=1).mean()

crls = nn.CrossEntropyLoss()

def train_refiner(model, dataloader, epochs):
    
    optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=5e-4)
        
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for x, label in dataloader:
            
            base_output = model(x)
            
            # Loss 1
            probs = F.gumbel_softmax(base_output, tau=1.0, hard=False)  # soft one-hot
            kl_loss = kl_divergence_loss(probs)
            loss = 1.0 * kl_loss + crls(base_output, label)
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        
        if (epoch+1)%10 == 0:
            print(f"Epoch {epoch+1}: Loss = {total_loss:.4f}")
            num_test = evaluate(model, evaluate_dataset)
            if num_test == 1024:
                break

def evaluate(model, dataset):
    model.eval()
        
    bucket_set = set()
    
    torch.manual_seed(99)
    
    with torch.no_grad():
        for x, label in dataset:
            
            x = x.unsqueeze(0)
            
            base_out = model(x)
            
            pred_bucket = torch.argmax(base_out, dim=1)[0].item()
            bucket_set.add(pred_bucket)
    
    print(len(bucket_set))
    
    return len(bucket_set)

##########################################################################################

loader = torch.utils.data.DataLoader(special_dataset, batch_size=4096, shuffle=True)

train_refiner(model, loader, epochs=1000)

torch.save(model.state_dict(), "fine_tuned_Dispatcher.pt")